# Running FastAPI in Google Colab



Welcome to FastAPI! We are exploring a modern, lightning-fast web framework for building APIs with Python based on standard type hints. Because FastAPI is designed to run as a continuous server, we are using tmux (a terminal multiplexer) right here in our Colab notebook to run the application in the background while keeping our notebook cells completely free to send requests and test our endpoints in real-time.

### References

- [FastAPI: Modern Python Web Development]( https://learning.oreilly.com/library/view/fastapi/9781098135492/ )
  - [companion repo]( https://github.com/madscheme/fastapi )
- [Building Generative AI Services with FastAPI]( https://learning.oreilly.com/library/view/building-generative-ai/9781098160296/ )
  - [companion repo]( https://github.com/Ali-Parandeh/building-generative-ai-services )






## Setup FastAPI



### Create scripts



FastAPI setup

In [ ]:
%%writefile tmux.fastapi.setup.sh
#!/bin/bash
mkdir -p fastapi
cd fastapi

# Create runner script.  This doesn't change.
cat <<'eof' > runner.py
import uvicorn

if __name__ == "__main__":
    # "fastapi_app:app" tells uvicorn to look in fastapi_app.py for the 'app' object
    # reload=True is the magic ingredient that reloads the service if the app changes
    uvicorn.run("fastapi_app:app", host="127.0.0.1", port=8000, reload=True)
eof

# Create app.  This can change and include other scripts.
cat <<'eof' > fastapi_app.py
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def read_root():
    return {"message": "Hello World! Try changing this text."}

eof


FastAPI run

In [ ]:
%%writefile tmux.fastapi.runner.sh
#!/bin/bash
python ./fastapi/runner.py
echo == Done
sleep 30


### Start tmux session and run scripts


In [ ]:
%%bash
# Start the tmux session
tmux new-session -s fastapi -d

# Run the setup script
tmux send-keys -t fastapi 'bash ./tmux.fastapi.setup.sh' Enter

sleep 2

# Run the runner script
tmux send-keys -t fastapi 'bash ./tmux.fastapi.runner.sh' Enter


## Enable port tunneling to view the FastAPI service.


In [ ]:
from datetime import datetime, timezone
from time import sleep
from google.colab import output
import requests

url = "http://127.0.0.1:8000"

print("Waiting for FastAPI to start")
for i in range(300):
  try:
    requests.head( url )
    print()
    break
  except:
    print("=", end="")
  sleep(1)
print(f"{i} seconds")

print("FastAPI has started")
output.serve_kernel_port_as_window(8000)


## Test using requests


In [ ]:
response = requests.get( url )
response


In [ ]:
response.text


In [ ]:
response_dict = response.json()
response_dict
